# 09. 추가 민감도 — 후보월 확장과 대칭 결과 변수 (P10)
> 계획서: `계획/P10_추가민감도_후보월확장과_대칭결과.md` · 코드: `code/extra_cnt.py` · 전체 계산은 P5·P7 과 같은 코드(`validate_cnt`, `robust_cnt`)를 그대로 씁니다. 01~08 의 결과와 노트북은 바꾸지 않습니다.

본 분석의 **두 가지 설계 비대칭**이 결론을 만든 것은 아닌지 확인하는 **추가 민감도 분석**입니다. 규칙(비교 기준·k=6·활동량 하한 Φ·감소 기준 n)과 요구불입금액 약화 문턱·규모 경계·업종군은 **P36에서 정한 값을 그대로 고정**하고 한 축씩만 바꿉니다. 결과를 본 뒤 규칙을 바꾸지 않습니다.

| 점검 | 질문 | 어떻게 바꾸나 |
|---|---|---|
| **A. 후보월 확장** | 후보월이 2024.06부터인 것은 '전년 비교(L=12)를 가정했던 때'의 범위가 남은 것이다. 직전 기간 비교(L=6)가 허용하는 2023.12부터 보면 결론이 같은가? | 첫 후보월을 e=17(2024.06)에서 e=11(2023.12)로. 끝(2025.06)은 신호 후 6개월 추적 때문에 그대로 |
| **B. 대칭 결과 변수** | 건수 신호는 '직전 대비'인데 요구불입금액 약화는 '전년 같은 달 대비'다. 둘을 같은 종류로 맞춰도 같은가? | 약화를 **신호 후 6개월 평균이 직전 구간 평균 대비 얼마나 변했나**의 하위 5%로 다시 정의 — B1: 직전 6개월 `[e-5, e]`, B2: 직전 12개월 `[e-11, e]` |
| **C. 둘 다** | A 와 B 를 함께 | 확장 후보월 + B1 / B2 |

**읽는 법(사전 규칙)**: RR 의 95% 신뢰구간 하한이 1보다 크면 연관 유지. 기준(본 분석)과의 차이는 '기준 대비 비'(같은 기업 재표집으로 짝지은 비)의 95% 범위가 1을 포함하면 구별되지 않는 것으로 읽습니다. B 는 **결과의 정의 자체가 바뀐 것**이라 RR 의 크기를 기준과 직접 견주지 않고 '연관이 유지되는가'를 봅니다. 신호 기업의 약 75%는 약화가 아니라는 점(적중 24.6%)은 정의를 바꿔도 그대로 읽습니다.

**화면 출력은 집계값뿐 · 5 미만 칸은 `<5` · 법인ID 없음.** 부트스트랩이 많아 몇 분 걸립니다(`CNT_BOOT` 로 줄일 수 있음, 기본 2,000). 01~08 이 끝나 있어야 합니다.

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)

import math
import detect_cnt as dc
import validate_cnt as vc
import robust_cnt as rb
import extra_cnt as ex

spec = json.load(open(WORK / "interim" / "detect_spec_v2.json", encoding="utf-8"))
rule = spec["rule"]
MODE, K, L, PHI2, N2 = rule["비교 기준"], int(rule["k"]), int(rule["L"]), float(rule["phi"]), float(rule["n_pct"])
B = int(os.environ.get("CNT_BOOT", "2000"))
SEED36, SEED35 = 20261301, 20261302        # 07 과 같은 시드: 기준 결과가 07 과 같게 재현되고, 같은 표본의 변형은 같은 기업 재표집으로 기준과 짝지어 비교한다

def sup(x):
    return cl.sup(int(round(x)))

def f_rr(rr, lo, hi):
    return "-" if rr is None or not np.isfinite(rr) else (f"{rr:.2f} ({lo:.2f}~{hi:.2f})" if np.isfinite(lo) else f"{rr:.2f}")

mk36, mk35 = masks["P36"], masks["P35"]
C5 = vc.fit_constants(panel, mk36, MODE, K, PHI2, N2)          # 규칙·문턱·경계는 P36 에서 고정 (05 와 같은 값)
D36, D35 = vc.prepare(panel, mk36, C5), vc.prepare(panel, mk35, C5)
assert int((D36["det"] >= 0).sum()) == spec["power"]["P36"]["N1"], "04의 탐지 기업 수와 다릅니다 (04를 다시 실행하세요)"
t0 = time.time()
BASE36 = rb.run_variant(D36, D36["F"], B, SEED36)
BASE35 = rb.run_variant(D35, D35["F"], B, SEED35)
OBJ36, OBJ35 = {"R": BASE36["R"], "D": D36}, {"R": BASE35["R"], "D": D35}
print(f"고정 규칙(v2): 비교 기준 {MODE}, k={K}, Φ={PHI2:.2f}, n={N2:.0f}%, 약화 문턱 θ_O1={C5['theta_o1']:.3f}")
print(f"기준 (36개월 법인): 신호 {BASE36['n_det']:,}곳, RR {f_rr(BASE36['R']['rr'], BASE36['R']['lo'], BASE36['R']['hi'])}   "
      f"| (35개월 법인만): 신호 {BASE35['n_det']:,}곳, RR {f_rr(BASE35['R']['rr'], BASE35['R']['lo'], BASE35['R']['hi'])}   ({time.time() - t0:.0f}초)")

## 1. A. 후보월 확장 — 2023.12부터 보면

본 분석의 후보월(신호를 처음 판정하는 달)은 **2024.06~2025.06(13개월)** 입니다. 이 범위는 비교 기준을 정하기 전에 '전년 같은 달 대비'(12개월 전 자료 필요)를 가정해 정했고, 직전 기간 비교로 정한 뒤에도 같은 범위를 썼습니다. 직전 6개월 비교만 보면 변화율은 **2023.12**부터 정의됩니다(최근 6개월 2023.07~12, 직전 6개월 2023.01~06). 신호 후 6개월을 추적해야 하므로 끝(2025.06)은 그대로입니다.

- **첫 후보월만 바꿉니다**. Φ·n·k·약화 문턱 θ_O1·규모 경계·업종군은 P36에서 정한 값을 그대로 씁니다(재산출하지 않음).
- 확장하면 이른 달(2023.12~2024.05)에 이미 줄고 있던 기업이 거기서 '처음 신호'로 잡히므로, 본 분석에서 첫 달(2024.06)에 몰려 있던 신호 기업이 어떻게 달라지는지도 봅니다.

In [ ]:
E_LO = K - 1 + L                                  # 직전 기간 비교가 허용하는 가장 이른 후보월 (k=6, L=6 이면 11 = 2023.12)
D36x, D35x = vc.prepare(panel, mk36, C5, e_lo=E_LO), vc.prepare(panel, mk35, C5, e_lo=E_LO)
months = list(range(E_LO, dc.E_MAX + 1))
ym = lambda e: f"{str(int(panel.months[e]))[:4]}.{str(int(panel.months[e]))[4:]}"
fm0, fmx = ex.first_detect_by_month(D36["det"], months), ex.first_detect_by_month(D36x["det"], months)
rows = [{"기준월": ym(e), "본 분석 후보월": "포함" if e >= 17 else "제외(확장에서만)", "본 분석: 처음 신호 기업": sup(fm0[e]) if e >= 17 else "-", "확장: 처음 신호 기업": sup(fmx[e])} for e in months]
cl.show_table(pd.DataFrame(rows), "기준월별 처음 신호가 잡힌 기업 수 (36개월 법인)", OUT / "nb09", "first_signal_by_month")
cmp36, cmp35 = ex.compare_detection(D36["det"], D36x["det"]), ex.compare_detection(D35["det"], D35x["det"])
rows = [{"표본": nm, "본 분석 신호 기업": sup(c["n_base"]), "확장 신호 기업": sup(c["n_ext"]), "확장에서만 새로 잡힘": sup(c["new"]),
         "더 이른 달로 앞당겨짐": sup(c["earlier"]), "그대로": sup(c["same"])} for nm, c in (("36개월 법인", cmp36), ("35개월 법인만", cmp35))]
cl.show_table(pd.DataFrame(rows), "후보월 확장 전후의 신호 기업", OUT / "nb09", "detection_extension")
print(f"위험집합·층은 확장 후보월로 다시 만들어집니다 (36개월 법인: 신호 단위 {int(D36x['units'].is_det.sum()):,}, 비교 단위 {sup((~D36x['units'].is_det).sum())})")

## 2. B. 대칭 결과 변수 — 직전 대비 직후로 맞추면

본 분석의 약화는 `g = (신호 후 6개월 평균 − 1년 전 같은 달 평균) ÷ 1년 전 같은 달 평균`의 하위 5%(전년 동기 대비)입니다. 이를 건수 신호와 같은 **직전 대비**로 바꿉니다.

| 이름 | 식 | 비고 |
|---|---|---|
| B1 | `(신호 후 6개월 평균 − 직전 6개월 평균) ÷ 직전 6개월 평균`, 직전 6개월 = `[e-5, e]` | 두 시점 모두 관측된 달끼리 짝지어 평균(본 분석 g 와 같은 규칙). 신호의 비교 달 `[e-11, e-6]` 과 겹치지 않음 |
| B2 | 같은 식, 직전 12개월 = `[e-11, e]` | 신호가 잡히는 구간에서 이미 줄어든 값에 덜 끌려가도록 기준을 길게. 각 구간 길이-1개월 이상 관측 |

약화 문턱은 **P36 판정 가능 전체에서 새 정의의 5번째 백분위수**(본 분석과 같은 규칙)이고, 35개월 법인에는 P36의 문턱을 그대로 씁니다. 비교 기업은 본 분석과 같은 층(기준월 × 규모 5분위 × 업종군)에서 골라 **모든 기업에 공통인 계절 효과는 상쇄**됩니다.

In [ ]:
SYM = {"B1": ((1, 6), (-5, 0), "직전 6개월 대비"), "B2": ((1, 6), (-11, 0), "직전 12개월 대비")}

def sym_outcome(Dp, Dq, key):
    """P36(Dp)에서 문턱을 정해 35개월 법인(Dq)에 그대로 적용한다. 반환: (G36, G35, θ)"""
    post, base, _ = SYM[key]
    G36 = ex.alt_outcome(Dp["Y"], Dp["er"], post, base)
    theta = vc.theta_low(G36, Dp["status"], Dp["er"], 5.0)
    return G36, ex.alt_outcome(Dq["Y"], Dq["er"], post, base), theta

RES = []          # 시나리오 결과 모음
def add(group, label, sample, res, theta=None):
    RES.append({"group": group, "label": label, "sample": sample, "res": res, "theta": theta})

t0 = time.time()
# A: 후보월 확장 (결과는 본 분석과 같은 전년 동기 대비)
add("A 후보월 확장", "후보월 2023.12~2025.06 (19개월)", "36개월 법인", ex.run_scenario(D36x, D36x["F"], B, SEED36, OBJ36))
add("A 후보월 확장", "후보월 2023.12~2025.06 (19개월)", "35개월 법인만", ex.run_scenario(D35x, D35x["F"], B, SEED35, OBJ35))
# B: 대칭 결과 (본 분석 후보월)
TH = {}
for key in SYM:
    G36, G35, th = sym_outcome(D36, D35, key)
    TH[("기본", key)] = th
    add("B 대칭 결과", SYM[key][2], "36개월 법인", ex.run_scenario(D36, D36["F"], B, SEED36, OBJ36, G36, th), th)
    add("B 대칭 결과", SYM[key][2], "35개월 법인만", ex.run_scenario(D35, D35["F"], B, SEED35, OBJ35, G35, th), th)
# C: 둘 다
for key in SYM:
    G36, G35, th = sym_outcome(D36x, D35x, key)
    TH[("확장", key)] = th
    add("C 둘 다", "확장 후보월 + " + SYM[key][2], "36개월 법인", ex.run_scenario(D36x, D36x["F"], B, SEED36, OBJ36, G36, th), th)
    add("C 둘 다", "확장 후보월 + " + SYM[key][2], "35개월 법인만", ex.run_scenario(D35x, D35x["F"], B, SEED35, OBJ35, G35, th), th)
print(f"시나리오 {len(RES)}개 계산 완료 ({time.time() - t0:.0f}초)")
print("약화 문턱 θ (P36 판정 가능 전체의 5번째 백분위수):", {f"{a}/{b}": round(v, 3) for (a, b), v in TH.items()}, f"| 본 분석 θ_O1 = {C5['theta_o1']:.3f}")

### 두 약화 정의가 같은 기업을 약화로 보는가

신호가 잡힌 기업(본 분석 후보월) 중 **전년 동기 대비 약화(본 분석)** 와 **B1(직전 6개월 대비 약화)** 가 얼마나 겹치는지 보는 표입니다. 겹침이 작으면 두 정의는 서로 다른 종류의 감소를 잡는다는 뜻이고, 그래도 RR 이 유지되면 연관이 정의 하나에 의존하지 않는다는 뜻입니다.

In [ ]:
G_b1, _, th_b1 = sym_outcome(D36, D35, "B1")
det = D36["det"]
idx = np.flatnonzero(det >= 0)
gy, gs = D36["G"][idx, det[idx]], G_b1[idx, det[idx]]
okb = ~np.isnan(gy) & ~np.isnan(gs)
wy, ws = gy[okb] <= C5["theta_o1"] + 1e-9, gs[okb] <= th_b1 + 1e-9
a_, b_, c_, d_ = int((wy & ws).sum()), int((wy & ~ws).sum()), int((~wy & ws).sum()), int((~wy & ~ws).sum())
cl.show_table(pd.DataFrame({"": ["B1 약화", "B1 비약화"], "전년 동기 대비 약화": [sup(a_), sup(c_)], "전년 동기 대비 비약화": [sup(b_), sup(d_)]}),
              f"신호 기업 {sup(okb.sum())}곳의 두 약화 정의 교차표 (36개월 법인)", OUT / "nb09", "outcome_crosstab")
if min(a_, b_) >= 5:
    print(f"전년 동기 대비 약화 중 B1 도 약화인 비율: {100 * a_ / (a_ + b_):.1f}%")
if min(a_, c_) >= 5:
    print(f"B1 약화 중 전년 동기 대비도 약화인 비율: {100 * a_ / (a_ + c_):.1f}%")
if min(a_, b_, c_) < 5:
    print("(교차표에 5 미만 칸이 있어 그 칸과 연결된 비율은 표시하지 않습니다.)")

## 3. 한눈에 보기

In [ ]:
def row(label, sample, group, res, base_res=None):
    R = res["R"]
    ok = np.isfinite(R["rr"])
    r = {"묶음": group, "시나리오": label, "표본": sample, "신호 기업": sup(res["n_det"]),
         "신호군 약화율 %": round(100 * R["p_det"], 1) if ok else None, "비교군 약화율 % (표준화)": round(100 * R["p_ctrl"], 1) if ok else None,
         "RR (95% 범위)": f_rr(R["rr"], R["lo"], R["hi"]), "기준 대비 비 (95% 범위)": f_rr(*res["ratio"]) if "ratio" in res else "-"}
    if "overlap" in res and res["overlap"]:
        r["본 분석과 최초 신호 일치(Jaccard)"] = round(res["overlap"]["jaccard"], 2)
    r["읽는 법"] = ("신호 30곳 미만 → 참고만" if res["descriptive"] else ("연관 유지 (범위 하한 > 1)" if ok and R["lo"] > 1 else "범위가 1을 포함하거나 정의 불가 → 연관이 확인되지 않음"))
    return r

rows = [row("본 분석 (기준)", "36개월 법인", "기준", BASE36), row("본 분석 (기준)", "35개월 법인만", "기준", BASE35)]
rows += [row(it["label"], it["sample"], it["group"], it["res"]) for it in RES]
TAB = pd.DataFrame(rows)
cl.show_table(TAB, "추가 민감도 결과 (같은 표본의 시나리오는 같은 기업 재표집으로 기준과 짝지은 비교)", OUT / "nb09", "scenarios")

fig, axes = plt.subplots(1, 2, figsize=(12, 0.42 * (len(RES) // 2 + 1) + 1.6), sharex=True)
for ax, smp, base in zip(axes, ("36개월 법인", "35개월 법인만"), (BASE36, BASE35)):
    items = [("본 분석 (기준)", base["R"], "기준")] + [(it["group"][0] + " " + it["label"], it["res"]["R"], it["group"]) for it in RES if it["sample"] == smp]
    col = {"기준": "#eb6834", "A": "#2a78d6", "B": "#1b9e77", "C": "#7570b3"}
    for i, (lab, R, g) in enumerate(items[::-1]):
        c = col.get(g[0], "#666")
        if np.isfinite(R["rr"]):
            ax.plot([R["lo"], R["hi"]], [i, i], color=c, lw=2.5, solid_capstyle="round"); ax.plot([R["rr"]], [i], "o", color=c, ms=6)
            ax.text(R["hi"] + 0.2, i, f"{R['rr']:.1f}", va="center", fontsize=8)
    ax.axvline(1, color="k", ls="--", lw=1); ax.axvline(base["R"]["rr"], color="#eb6834", ls=":", lw=1)
    ax.set_yticks(range(len(items))); ax.set_yticklabels([x[0][:38] for x in items[::-1]], fontsize=8); ax.set_title(smp)
    ax.set_xlabel("RR (점 = 추정, 선 = 95% 범위, 검은 점선 = 1, 주황 점선 = 기준)")
plt.tight_layout(); plt.show()
vr = [it for it in RES if not it["res"]["descriptive"] and np.isfinite(it["res"]["R"]["rr"])]
lows = [f"{it['sample']} · {it['label']}" for it in vr if not (it["res"]["R"]["lo"] > 1)]
print(f"시나리오 {len(vr)}개의 RR 은 {min(it['res']['R']['rr'] for it in vr):.2f} ~ {max(it['res']['R']['rr'] for it in vr):.2f} 범위 (기준 36개월 {BASE36['R']['rr']:.2f}, 35개월만 {BASE35['R']['rr']:.2f}).")
print("95% 범위가 1을 포함하는(연관이 확인되지 않는) 시나리오:", lows if lows else "없음")

## 4. 요약 저장

In [ ]:
summary = {"rule": {"mode": MODE, "k": K, "phi": PHI2, "n_pct": N2, "theta_o1": float(C5["theta_o1"]), "e_lo_base": K + 11, "e_lo_ext": int(E_LO)},
           "base": {"P36": ex.pk(BASE36), "P35": ex.pk(BASE35)},
           "scenarios": [{"group": it["group"], "label": it["label"], "sample": it["sample"], "theta": it["theta"], **ex.pk(it["res"])} for it in RES],
           "first_signal_by_month": {"base": {str(e): int(v) for e, v in fm0.items() if e >= 17}, "extended": {str(e): int(v) for e, v in fmx.items()}},
           "detection": {"P36": cmp36, "P35": cmp35},
           "crosstab_yoy_vs_b1": {"both": a_, "yoy_only": b_, "b1_only": c_, "neither": d_},
           "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "nb09_summary.json", summary)
print("저장: interim/nb09_summary.json,", "output/nb09/*.csv")

## 체크포인트 (Claude에게 알려줄 것)
공유 가능한 범위(팀 확인)에서 아래만 알려주세요.
- [ ] 본 분석 기준 RR 이 05·07 과 같게 재현되는가 (36개월 법인 4.69, 35개월 법인만 4.29 근처)
- [ ] **A**: 확장 후보월에서 신호 기업 수, 확장에서만 새로 잡힌 기업·앞당겨진 기업 수, RR(95% 범위)과 기준 대비 비, 기준월별 처음 신호 기업 수(2024.06 쏠림이 퍼지는가)
- [ ] **B1·B2**: 약화 문턱 θ, 신호군·비교군 약화율, RR(95% 범위)과 기준 대비 비, 두 약화 정의의 교차표(겹침 비율)
- [ ] **C**: 둘 다 적용한 RR(95% 범위)
- [ ] 95% 범위가 1을 포함하는 시나리오가 있는가 (35개월 법인만 포함)

**다음**: 계획서 `P10` 판정 기록 표에 결과와 해석(가설)을 적고, 결과 문서의 한계·Q&A 문구를 갱신합니다.